In [37]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import plotly.express as px
import numpy as np

from utils import get_data

seed = 42
wine_X_tr, wine_X_val, wine_X_test, wine_y_tr, wine_y_val, wine_y_test = get_data(seed)

In [ ]:
from typing import Any

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from sklearn.discriminant_analysis import unique_labels


class NN(BaseEstimator, ClassifierMixin):
    def __init__(self, hidden_dims=[100], dropout=0.0, epochs=200, batch_size=128, 
                 early_stopping=True, patience=10, tol=1e-4, random_state=seed, **kwargs):
        self.hidden_dims = hidden_dims
        self.dropout = dropout
        self.epochs = epochs
        self.batch_size = batch_size
        self.early_stopping = early_stopping
        self.patience = patience
        self.tol = tol
        self.random_state = random_state
        self.kwargs = kwargs
        
    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        X, y = check_X_y(X, y)
        self.classes_ = unique_labels(y)
        self.num_features_ = X.shape[1]
        
        y_mapped = np.searchsorted(self.classes_, y)

        layer_dims = [self.num_features_, *self.hidden_dims, len(self.classes_)]
        layers = []
        for i in range(len(layer_dims)-2):
            layers.append(nn.Linear(layer_dims[i], layer_dims[i+1]))
            layers.append(nn.ReLU())
            if self.dropout:
                layers.append(nn.Dropout(self.dropout))
        layers.append(nn.Linear(layer_dims[-2], layer_dims[-1]))

        self.model_ = nn.Sequential(*layers)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.model_.parameters(), **self.kwargs)

        self.model_.train()
        X_tensor = torch.as_tensor(X, dtype=torch.float32)
        y_tensor = torch.as_tensor(y_mapped, dtype=torch.long)
        
        n_samples = len(X)
        best_loss = float('inf')
        no_improve = 0

        for epoch in range(self.epochs):
            perm = torch.randperm(n_samples)
            X_shuffled = X_tensor[perm]
            y_shuffled = y_tensor[perm]
            
            epoch_loss = 0.0
            for i in range(0, n_samples, self.batch_size):
                batch_X = X_shuffled[i : i + self.batch_size]
                batch_y = y_shuffled[i : i + self.batch_size]
                optimizer.zero_grad()
                outputs = self.model_(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
            if self.early_stopping:
                if epoch_loss + self.tol < best_loss:
                    best_loss = epoch_loss
                    no_improve = 0
                else:
                    no_improve += 1
                    if no_improve >= self.patience:
                        break

        return self

    def predict_proba(self, X):
        check_is_fitted(self)
        X = check_array(X)

        self.model_.eval()
        with torch.inference_mode():
            X_tensor = torch.as_tensor(X, dtype=torch.float32)
            outputs = self.model_(X_tensor)
            probs = torch.softmax(outputs, dim=1)

        return probs.numpy()
    
    def predict(self, X):
        probs = self.predict_proba(X)
        return self.classes_[np.argmax(probs, axis=1)]

def plotly_plot(lines: list[list[tuple[Any, Any]]], labels: list[str], x_name: str = "x axis", y_name: str = "y axis", label_name: str = "label", **kwargs):
    df = pd.DataFrame([{x_name: x, y_name: y, label_name: label} for data, label in zip(lines, labels) for x, y in data])
    fig = px.line(df, x=x_name, y=y_name, color=label_name, **kwargs)
    fig.show()

def train_and_plot(params_list: list[dict], param_to_test: str, train_sizes=np.linspace(0.1, 1.0, 15)):
    data_size = len(wine_X_tr)
    train_accuracies = []
    val_accuracies = []
    for params in params_list:
        train_acc_curve = []
        val_acc_curve = []
        np.random.seed(seed)
        for train_size in train_sizes:
            model = NN(**params)
            indices = np.random.choice(data_size, int(train_size * data_size), replace=False)
            wine_X_tr_batch = wine_X_tr[indices]
            wine_y_tr_batch = wine_y_tr.iloc[indices]
            model.fit(wine_X_tr_batch, wine_y_tr_batch)
            train_acc_curve.append((train_size, model.score(wine_X_tr_batch, wine_y_tr_batch)))
            val_acc_curve.append((train_size, model.score(wine_X_val, wine_y_val)))
        train_accuracies.append(train_acc_curve)
        val_accuracies.append(val_acc_curve)

    plotly_plot(
        train_accuracies, [str(params[param_to_test]) for params in params_list], "train_size", "training accuracy", param_to_test
    )
    plotly_plot(
        val_accuracies, [str(params[param_to_test]) for params in params_list], "train_size", "validation accuracy", param_to_test
    )

In [39]:
train_and_plot(
    [
        {"hidden_dims": [16]},
        {"hidden_dims": [32]},
        {"hidden_dims": [64]},
        {"hidden_dims": [128]},
        {"hidden_dims": [16, 16]},
        {"hidden_dims": [32, 32]},
        {"hidden_dims": [64, 64]},
        {"hidden_dims": [128, 128]},
    ],
    "hidden_dims"
)

In [ ]:
train_and_plot(
    [
        {"lr": 0.0001},
        {"lr": 0.0005},
        {"lr": 0.001},
        {"lr": 0.005},
        {"lr": 0.01},
        {"lr": 0.05},
        {"lr": 0.1},
        {"lr": 0.5},
        {"lr": 1},
    ],
    "lr"
)

In [41]:
train_and_plot(
    [
        {"weight_decay": 0},
        {"weight_decay": 0.00001},
        {"weight_decay": 0.00005},
        {"weight_decay": 0.0001},
        {"weight_decay": 0.0005},
        {"weight_decay": 0.001},
        {"weight_decay": 0.005},
        {"weight_decay": 0.01},
        {"weight_decay": 0.05},
        {"weight_decay": 0.1},
    ],
    "weight_decay"
)

In [42]:
train_and_plot(
    [
        {"dropout": 0},
        {"dropout": 0.1},
        {"dropout": 0.2},
        {"dropout": 0.3},
        {"dropout": 0.4},
        {"dropout": 0.5},
        {"dropout": 0.6},
        {"dropout": 0.7},
    ],
    "dropout"
)